# Layer 6 — Agent Orchestration: Quality Tests

Validates the complete Layer 6 pipeline output across 7 assertions.
Run **after** executing either `6.2_agent_orchestrator.py` or `6.3_agent_runner.py`.

| Script | Output | Key validation |
|---|---|---|
| `6.2_agent_orchestrator.py` | `data/agent_run_log.json` | Status, tool trace, tool count |
| `6.2_agent_orchestrator.py` | `outputs/executive_summary_{date}.txt` | Word count, required sections |
| `6.2_agent_orchestrator.py` | `data/agent_results.csv` | Row count, column completeness |
| `6.3_agent_runner.py` | All of the above | End-to-end via CLI entry point |

In [1]:
import json
import os
import pandas as pd
from pathlib import Path

DATA    = Path("../data")
OUTPUTS = Path("../outputs")

LOG_PATH = DATA / "agent_run_log.json"

assert LOG_PATH.exists(), (
    f"agent_run_log.json not found. Run 6.2_agent_orchestrator.py or "
    f"6.3_agent_runner.py first."
)

with open(LOG_PATH, encoding="utf-8") as f:
    log = json.load(f)

run_date   = log["date"]
tool_trace = log["tool_trace"]
tools_used = log["tools_used"]

ar = pd.read_csv(DATA / "anomaly_results.csv")
dl = pd.read_csv(DATA / "delivery_log.csv")
ag = pd.read_csv(DATA / "agent_results.csv") if (DATA / "agent_results.csv").exists() else None

print(f"agent_run_log.json loaded")
print(f"  Run date    : {run_date}")
print(f"  Status      : {log['status']}")
print(f"  Tool calls  : {log['tool_call_count']}")
print(f"  Tools used  : {tools_used}")
print(f"  Model       : {log['model']}")

agent_run_log.json loaded
  Run date    : 2024-08-20
  Status      : completed
  Tool calls  : 20
  Tools used  : ['fetch_kpis', 'run_detection', 'run_rca', 'score_impact', 'prioritize', 'lookup_playbook', 'send_alert', 'generate_executive_summary', 'update_powerbi_dataset']
  Model       : claude-sonnet-4-6


---
## Test 1 — Run Log Exists and Pipeline Completed
The run log must exist, report `status = 'completed'`, and contain at least
9 tool calls (one per pipeline step) across at least 2 agent turns.

In [2]:
assert LOG_PATH.exists(),              f"agent_run_log.json not found"
assert log["status"] == "completed",  f"Pipeline did not complete: status={log['status']}"
assert log["tool_call_count"] >= 9,   f"Fewer than 9 tool calls: {log['tool_call_count']}"
assert log["total_turns"] >= 2,       f"Fewer than 2 turns: {log['total_turns']}"
assert log["model"] == "claude-sonnet-4-6", f"Unexpected model: {log['model']}"

print(f"PASS  Run log exists         : {LOG_PATH}")
print(f"PASS  Status                 : {log['status']}")
print(f"PASS  Tool calls             : {log['tool_call_count']} (>= 9)")
print(f"PASS  Agent turns            : {log['total_turns']} (>= 2)")
print(f"PASS  Model                  : {log['model']}")

PASS  Run log exists         : ..\data\agent_run_log.json
PASS  Status                 : completed
PASS  Tool calls             : 20 (>= 9)
PASS  Agent turns            : 7 (>= 2)
PASS  Model                  : claude-sonnet-4-6


---
## Test 2 — All 9 Tool Names Present
Every tool in the defined set must appear at least once in `tools_used`.
A missing tool means the agent skipped a pipeline step.

In [3]:
REQUIRED_TOOLS = [
    "fetch_kpis",
    "run_detection",
    "run_rca",
    "score_impact",
    "prioritize",
    "lookup_playbook",
    "send_alert",
    "generate_executive_summary",
    "update_powerbi_dataset",
]

missing = [t for t in REQUIRED_TOOLS if t not in tools_used]
assert len(missing) == 0, f"Tools missing from run: {missing}"

print(f"{'Tool':<40}  Calls")
print("-" * 48)
for t in REQUIRED_TOOLS:
    n = sum(1 for e in tool_trace if e["tool_name"] == t)
    print(f"PASS  {t:<38}  {n:>2}")

Tool                                      Calls
------------------------------------------------
PASS  fetch_kpis                               1
PASS  run_detection                            1
PASS  run_rca                                  3
PASS  score_impact                             3
PASS  prioritize                               1
PASS  lookup_playbook                          3
PASS  send_alert                               6
PASS  generate_executive_summary               1
PASS  update_powerbi_dataset                   1


---
## Test 3 — Decision Flow Order Respected
The agent must call tools in the correct causal order:
fetch → detect → rca/impact → prioritize → playbook → alert → summary → powerbi.
`update_powerbi_dataset` must be the absolute last tool call.

In [4]:
def first_step(tool_name):
    steps = [e["step"] for e in tool_trace if e["tool_name"] == tool_name]
    return min(steps) if steps else float("inf")

ORDER_CONSTRAINTS = [
    ("fetch_kpis",                 "run_detection"),
    ("run_detection",              "run_rca"),
    ("run_detection",              "score_impact"),
    ("prioritize",                 "lookup_playbook"),
    ("lookup_playbook",            "send_alert"),
    ("send_alert",                 "generate_executive_summary"),
    ("generate_executive_summary", "update_powerbi_dataset"),
]

for earlier, later in ORDER_CONSTRAINTS:
    e_step = first_step(earlier)
    l_step = first_step(later)
    assert e_step < l_step, (
        f"Order violated: {earlier} (step {e_step}) must precede {later} (step {l_step})"
    )
    print(f"PASS  {earlier:<35}  before  {later}")

last_tool = tool_trace[-1]["tool_name"]
assert last_tool == "update_powerbi_dataset", (
    f"Last tool was {last_tool!r}, expected 'update_powerbi_dataset'"
)
print(f"\nPASS  update_powerbi_dataset is the final tool call (step {tool_trace[-1]['step']})")

PASS  fetch_kpis                           before  run_detection
PASS  run_detection                        before  run_rca
PASS  run_detection                        before  score_impact
PASS  prioritize                           before  lookup_playbook
PASS  lookup_playbook                      before  send_alert
PASS  send_alert                           before  generate_executive_summary
PASS  generate_executive_summary           before  update_powerbi_dataset

PASS  update_powerbi_dataset is the final tool call (step 20)


---
## Test 4 — Executive Summary Saved and >= 200 Words
The orchestrator must save `outputs/executive_summary_{date}.txt` with at least
200 words and all three required structural sections.

In [5]:
summary_path = OUTPUTS / f"executive_summary_{run_date}.txt"
assert summary_path.exists(), f"Executive summary not found: {summary_path}"

text       = summary_path.read_text(encoding="utf-8")
word_count = len(text.split())
assert word_count >= 200, f"Summary too short: {word_count} words (minimum 200)"

REQUIRED_SECTIONS = [
    "SITUATION OVERVIEW",
    "HIGH PRIORITY",
    "RECOMMENDED IMMEDIATE ACTIONS",
]
for section in REQUIRED_SECTIONS:
    assert section in text, f"Missing section in summary: {section!r}"

print(f"PASS  Summary file     : {summary_path.name}")
print(f"PASS  Word count       : {word_count} words (>= 200)")
for section in REQUIRED_SECTIONS:
    print(f"PASS  Section present  : {section!r}")

# Show first 3 lines as a sanity preview
preview = "\n".join(text.splitlines()[:3])
print(f"\nPreview:\n{preview}")

PASS  Summary file     : executive_summary_2024-08-20.txt
PASS  Word count       : 789 words (>= 200)
PASS  Section present  : 'SITUATION OVERVIEW'
PASS  Section present  : 'HIGH PRIORITY'
PASS  Section present  : 'RECOMMENDED IMMEDIATE ACTIONS'

Preview:
DAILY KPI ANOMALY BRIEF - 2024-08-20



---
## Test 5 — Anomaly Count Matches Layer 2 Ground Truth
The count returned by `run_detection` in the agent trace must match the
row count for the same date in `anomaly_results.csv` (Layer 2 output).
A mismatch would indicate the agent read stale or incorrect data.

In [6]:
det_entry = next(
    (e for e in tool_trace if e["tool_name"] == "run_detection"), None
)
assert det_entry is not None, "run_detection not found in tool_trace"

agent_count    = det_entry["output"].get("count", -1)
agent_severity = det_entry["output"].get("severity_summary", {})

l2_rows     = ar[ar["date"] == run_date]
l2_count    = len(l2_rows)
l2_severity = l2_rows["severity"].value_counts().to_dict()

assert agent_count == l2_count, (
    f"Count mismatch: agent={agent_count}, Layer 2={l2_count}"
)

print(f"PASS  Anomaly count     : agent={agent_count}  Layer 2={l2_count}  (exact match)")
print(f"PASS  Severity (agent)  : {agent_severity}")
print(f"PASS  Severity (L2)     : {l2_severity}")

PASS  Anomaly count     : agent=6  Layer 2=6  (exact match)
PASS  Severity (agent)  : {'LOW': 3, 'HIGH': 2, 'MEDIUM': 1}
PASS  Severity (L2)     : {'LOW': 3, 'HIGH': 2, 'MEDIUM': 1}


---
## Test 6 — Alert Routing Accuracy
For the run date, `delivery_log.csv` must show correct channel and status
per severity tier:
- HIGH (unsuppressed)   → Slack + Email  → SENT
- MEDIUM (unsuppressed) → Email          → QUEUED
- LOW                   → Digest         → SCHEDULED

In [7]:
dl_date    = dl[dl["date"] == run_date].copy()
ar_date    = ar[ar["date"] == run_date][["anomaly_id", "severity"]]
dl_merged  = dl_date.merge(ar_date, on="anomaly_id")

assert len(dl_merged) == l2_count, (
    f"delivery_log row count ({len(dl_merged)}) != anomaly count ({l2_count})"
)

ROUTING = {
    "HIGH":   ("Slack + Email", "SENT"),
    "MEDIUM": ("Email",         "QUEUED"),
    "LOW":    ("Digest",        "SCHEDULED"),
}

suppressed = dl_merged[dl_merged["delivery_status"] == "SUPPRESSED"]

for severity, (exp_channel, exp_status) in ROUTING.items():
    active = dl_merged[
        (dl_merged["severity"] == severity) &
        (dl_merged["delivery_status"] != "SUPPRESSED")
    ]
    if active.empty:
        print(f"INFO  {severity:<8}: no unsuppressed anomalies on {run_date} — skip")
        continue

    bad_ch = active[active["delivery_channel"] != exp_channel]
    bad_st = active[active["delivery_status"]  != exp_status]

    assert bad_ch.empty, (
        f"{severity}: wrong channel — expected {exp_channel!r}, "
        f"got {bad_ch['delivery_channel'].unique().tolist()}"
    )
    assert bad_st.empty, (
        f"{severity}: wrong status — expected {exp_status!r}, "
        f"got {bad_st['delivery_status'].unique().tolist()}"
    )
    print(
        f"PASS  {severity:<8}  ({len(active)} anomaly/ies)  "
        f"channel={exp_channel:<20}  status={exp_status}"
    )

if len(suppressed) > 0:
    print(
        f"PASS  {len(suppressed)} suppressed anomaly/ies — "
        f"correctly withheld from executive channel"
    )
else:
    print(f"INFO  No suppressed alerts on {run_date}")

PASS  HIGH      (2 anomaly/ies)  channel=Slack + Email         status=SENT
PASS  MEDIUM    (1 anomaly/ies)  channel=Email                 status=QUEUED
PASS  LOW       (3 anomaly/ies)  channel=Digest                status=SCHEDULED
INFO  No suppressed alerts on 2024-08-20


---
## Test 7 — Power BI Export Written and Complete
`data/agent_results.csv` must exist, contain rows for the run date, match
the detected anomaly count, and include all key downstream columns.

In [8]:
ag_path = DATA / "agent_results.csv"
assert ag_path.exists(), f"agent_results.csv not found at {ag_path}"

ag      = pd.read_csv(ag_path)
ag_date = ag[ag["date"] == run_date]

assert len(ag_date) > 0, f"No rows for {run_date} in agent_results.csv"
assert len(ag_date) == l2_count, (
    f"Row count mismatch: agent_results={len(ag_date)}, detected anomalies={l2_count}"
)

REQUIRED_COLS = [
    "anomaly_id", "date", "kpi", "tier", "severity",
    "priority_rank", "revenue_at_risk", "delivery_status",
    "alert_subject", "recommended_owner",
]
missing_cols = [c for c in REQUIRED_COLS if c not in ag.columns]
assert len(missing_cols) == 0, f"Missing columns in agent_results.csv: {missing_cols}"

print(f"PASS  agent_results.csv exists    : {ag_path}")
print(f"PASS  Rows for {run_date}         : {len(ag_date)}")
print(f"PASS  Row count matches detection : {len(ag_date)} == {l2_count}")
print(f"PASS  Required columns present    : {len(REQUIRED_COLS)} columns verified")
for col in REQUIRED_COLS:
    print(f"        {col}")

PASS  agent_results.csv exists    : ..\data\agent_results.csv


PASS  Rows for 2024-08-20         : 6
PASS  Row count matches detection : 6 == 6
PASS  Required columns present    : 10 columns verified
        anomaly_id
        date
        kpi
        tier
        severity
        priority_rank
        revenue_at_risk
        delivery_status
        alert_subject
        recommended_owner


---
## All 7 Tests — Summary

| # | Test | Source | Expected |
|---|---|---|---|
| T01 | Run log exists and pipeline completed | `agent_run_log.json` | status=completed, >= 9 tool calls, >= 2 turns |
| T02 | All 9 tool names present | `agent_run_log.json` | All tools in tools_used list |
| T03 | Decision flow order respected | `agent_run_log.json` | fetch before detect, summary before powerbi, powerbi last |
| T04 | Executive summary saved and >= 200 words | `outputs/executive_summary_{date}.txt` | >= 200 words, 3 required sections |
| T05 | Anomaly count matches Layer 2 | `anomaly_results.csv` | Exact count match |
| T06 | Alert routing accuracy | `delivery_log.csv` | HIGH→Slack+Email, MEDIUM→Email, LOW→Digest |
| T07 | Power BI export written and complete | `agent_results.csv` | Rows for date, 10 required columns |

In [9]:
print("All 7 Layer 6 quality tests passed.")
print(f"Agent orchestration for {run_date} is certified complete.")
print("agent_results.csv is ready for Power BI consumption.")

All 7 Layer 6 quality tests passed.
Agent orchestration for 2024-08-20 is certified complete.
agent_results.csv is ready for Power BI consumption.
